# Embedding dot-product book rating prediction

Explicit ratings only. Evaluation reports known-user/item coverage; cold-start pairs are excluded and are not covered by the reported score.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data"
OUT = ROOT / "artifacts"
OUT.mkdir(exist_ok=True)
keras.utils.set_random_seed(42)
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Model

import matplotlib.pyplot as plt


In [ ]:
ratings = pd.read_csv(DATA / "Ratings.csv", dtype={"ISBN": str})
# Zero denotes implicit feedback; this experiment predicts explicit ratings only.
ratings = ratings.loc[ratings["Book-Rating"].between(1, 10)].dropna(subset=["User-ID", "ISBN"])
ratings = ratings.groupby(["User-ID", "ISBN"], as_index=False)["Book-Rating"].mean()
train_raw, test_raw = train_test_split(ratings, test_size=0.1, random_state=42)
train_raw, val_raw = train_test_split(train_raw, test_size=0.1, random_state=42)
users_new_id = {u: i+1 for i, u in enumerate(sorted(train_raw["User-ID"].unique()))}
books_new_id = {b: i+1 for i, b in enumerate(sorted(train_raw["ISBN"].unique()))}
def encode(frame):
    known = frame["User-ID"].isin(users_new_id) & frame["ISBN"].isin(books_new_id)
    print(f"Known-user/item coverage: {known.mean():.1%} ({known.sum()}/{len(frame)})")
    frame = frame.loc[known]
    if frame.empty:
        raise ValueError("No known user-item pairs in this partition; use a larger dataset.")
    return pd.DataFrame({0: frame["User-ID"].map(users_new_id), 1: frame["ISBN"].map(books_new_id), 2: frame["Book-Rating"]}).reset_index(drop=True)
train, validation, test = map(encode, [train_raw, val_raw, test_raw])
assert set(zip(train_raw["User-ID"], train_raw.ISBN)).isdisjoint(zip(test_raw["User-ID"], test_raw.ISBN))

In [ ]:
user_input_tensor = keras.layers.Input(shape=[1],name='user-input')
user_embedding = keras.layers.Embedding(len(users_new_id)+1, 10, 
                    name='user-embedding')(user_input_tensor)
user_vector = keras.layers.Flatten(name='user-flatten')(user_embedding) 

In [ ]:
book_input_tensor = keras.layers.Input(shape=[1],name='book-input')
book_embedding = keras.layers.Embedding(len(books_new_id)+1, 10, 
                    name='book-embedding')(book_input_tensor)
book_vector = keras.layers.Flatten(name='book-flatten')(book_embedding) 

In [ ]:
print(book_vector.shape)
print(user_vector.shape)

In [ ]:
d = keras.layers.Dot(axes=1, normalize=True)([book_vector, user_vector])
dense_1 = keras.layers.Dense(64, name='dense_1', activation='relu')(d)
dense_2= keras.layers.Dense(32, name= 'dense_2', activation='relu')(dense_1)
dense_3 = keras.layers.Dense(16, name='dense_3', activation='relu')(dense_2)
output_layer = keras.layers.Dense(1,name='output_layer', activation='relu')(dense_3)
My_model = Model([user_input_tensor, book_input_tensor], output_layer)

In [ ]:
My_model.compile(loss='mean_absolute_error', optimizer='adam')

In [ ]:
My_model.summary()

In [ ]:
tf.config.run_functions_eagerly(False)
fitModel = My_model.fit([train[0], train[1]], train[2], epochs=20, validation_data=([validation[0], validation[1]], validation[2]), callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

In [ ]:
pd.Series(fitModel.history['loss']).plot(logy=True)
plt.xlabel("Epochs")
plt.ylabel("Train Error")
plt.show()

In [ ]:
predicted = My_model.predict([test[0], test[1]])
results = {"mae": mean_absolute_error(test[2], predicted), "rmse": float(np.sqrt(mean_squared_error(test[2], predicted))), "train_median_baseline_mae": mean_absolute_error(test[2], np.full(len(test), train[2].median()))}
print(results)
(OUT / "dot_metrics.json").write_text(json.dumps(results, indent=2))

In [ ]:
My_model.save(OUT / "dot_model.keras")
(OUT / "dot_mappings.json").write_text(json.dumps({"users": {str(k): int(v) for k, v in users_new_id.items()}, "books": books_new_id}, indent=2))